# Task 2 — Battery SoC prediction & BMS anomaly detection

This notebook mirrors the assignment deliverables:
- Feature engineering + target: SoC at **t+10 min** (30s cadence → 20 steps).
- **Model A**: XGBoost with **TimeSeriesSplit** CV; test RMSE/MAE/R²; **SHAP** summary.
- **Model B**: **LSTM** (2 layers, dropout 0.2, seq len 10); test RMSE/MAE; pred vs actual for top devices.
- **Anomalies**: rule flags + **IsolationForest** confidence → `anomaly_flags.csv`.

## Why IsolationForest (vs Autoencoder)

- Works well on tabular telemetry features with mixed scales after standardization.
- Robust, fast to train, and easy to score online.
- Produces an anomaly score that we convert into a **0–1 confidence** value.

Run the pipeline (same as `python task2_soc_model.py`):

In [ ]:
import runpy

runpy.run_path("task2_soc_model.py", run_name="__main__")

## Outputs

After the run completes, the artifacts are written to:

- `outputs/task2/soc_features.csv`
- `outputs/task2/xgboost_metrics.json`
- `outputs/task2/shap_xgboost_summary.png`
- `outputs/task2/lstm_metrics.json`
- `outputs/task2/lstm_pred_vs_actual_<device>.png`
- `outputs/task2/model_comparison_table.csv`
- `outputs/task2/anomaly_flags.csv`

The next cells load and render the comparison table and key metrics.

In [ ]:
import json
from pathlib import Path

import pandas as pd

out_dir = Path("outputs/task2")

comparison = pd.read_csv(out_dir / "model_comparison_table.csv")
comparison

In [ ]:
with (out_dir / "xgboost_metrics.json").open("r", encoding="utf-8") as f:
    xgb_metrics = json.load(f)
with (out_dir / "lstm_metrics.json").open("r", encoding="utf-8") as f:
    lstm_metrics = json.load(f)

pd.DataFrame(
    {
        "metric": ["test_rmse", "test_mae", "test_r2", "training_time_sec", "inference_latency_ms_per_sample"],
        "xgboost": [
            xgb_metrics.get("test_rmse"),
            xgb_metrics.get("test_mae"),
            xgb_metrics.get("test_r2"),
            xgb_metrics.get("training_time_sec"),
            xgb_metrics.get("inference_latency_ms_per_sample"),
        ],
        "lstm": [
            lstm_metrics.get("test_rmse"),
            lstm_metrics.get("test_mae"),
            None,
            lstm_metrics.get("training_time_sec"),
            lstm_metrics.get("inference_latency_ms_per_sample"),
        ],
    }
)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(out_dir / "shap_xgboost_summary.png")))

In [ ]:
plots = sorted(out_dir.glob("lstm_pred_vs_actual_*.png"))
for p in plots[:3]:
    display(Image(filename=str(p)))

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(out_dir / "model_comparison_table.png")))